# 0. 各項目がどういうデータか調べる

予測を始める前に、**手元のデータに何が入っているのか**を確かめる。

項目の名前だけ見ても中身は分からない。`sibsp` や `parch` のように、
名前からは何の数字か想像できないものもある。

この章では、気づいた疑問を順に追っていく。

1. 各項目がどんな値を取るのか
2. `sibsp` と `parch` は何を数えた数字なのか
3. 家族構成を知る手がかりが他の項目にないか
4. `fare`（運賃）は一人分の料金なのか、家族の合計なのか
5. 運賃を手がかりにすると、家族がどこまで見えるか
6. 性別・年齢・等級・港も合わせると、家族構成がどこまで分かるか
7. 分からない部分があっても、家族構成を項目として使えるか

**どれも「この項目をそのまま使ってよいか」を判断するための材料になる。**

## 1. データを読み込む

`data/raw/train.csv` を読み込んで、大きさを確認する。

In [ ]:
import pandas as pd

train = pd.read_csv("../../data/raw/train.csv", index_col=0)

print(f"train: {train.shape[0]} 人 × {train.shape[1]} 項目")

## 2. 項目の意味

8つの項目の意味は、データの説明ページに次のように書かれている。

| 項目 | 意味 |
| --- | --- |
| `survived` | 生還したか（1=生還, 0=死亡）。**これを予測する** |
| `pclass` | 客室の等級（1 が最上級, 3 が最下級） |
| `sex` | 性別 |
| `age` | 年齢 |
| `sibsp` | 同乗した兄弟姉妹・配偶者の数 |
| `parch` | 同乗した親・子の数 |
| `fare` | 運賃 |
| `embarked` | 乗った港（S=サウサンプトン, C=シェルブール, Q=クイーンズタウン） |

`id` が項目数に入っていないのは、`index_col=0` で行の名前として扱っているため。

**書かれているのはこれだけ。** この後で調べるのは、ここから分からない部分になる。

## 3. 各項目が取りうる値

どんな値が入っているのかを一覧にする。

種類が少ない項目は全部並べ、多い項目は最小と最大だけ出す。
値が入っていない人がいる項目には、その件数を添える。

In [ ]:
for col in train.columns:
    values = train[col]
    n_missing = values.isna().sum()

    # 種類が 10 以下なら全部並べる。多い項目は最小〜最大だけにする
    if values.nunique() <= 10:
        info = ", ".join(map(str, sorted(values.dropna().unique())))
    else:
        info = f"{values.min()} 〜 {values.max()}（{values.nunique()} 種類）"

    print(f"  {col:9} {info}" + (f"   ※ 空き {n_missing} 件" if n_missing else ""))

### ここで気づくこと

**すぐ対応が必要なもの**

- `age` に 85 件、`embarked` に 2 件、値が入っていない人がいる
- `sex` と `embarked` は文字。`female` / `male` のままでは計算に使えない

**気になるが、まだ理由が分からないもの**

- **`sibsp` は 0〜5 と 8 で、6 と 7 が飛んでいる** — なぜ抜けているのか
- `fare` が 0.0 から始まる — 無料で乗った人がいる

`sibsp` の欠けは後で追う。まずはこの項目が何を数えているのかを押さえる。

## 4. `sibsp` と `parch` は何を数えた数字か

どちらも**同乗していた家族の人数**だが、家族の中での関係によって分類が変わる。

| 項目 | 数える相手 |
| --- | --- |
| `sibsp` | **兄弟姉妹** と **配偶者** |
| `parch` | **親** と **子** |

サザエさんの家族7人が全員乗ったとすると、こうなる。

| 人 | `sibsp` | 内訳 | `parch` | 内訳 |
| --- | --- | --- | --- | --- |
| 波平 | 1 | フネ（妻） | 3 | サザエ・カツオ・ワカメ |
| フネ | 1 | 波平（夫） | 3 | サザエ・カツオ・ワカメ |
| サザエ | 3 | マスオ（夫）+ カツオ・ワカメ | 3 | 波平・フネ + タラオ |
| カツオ | 2 | サザエ・ワカメ | 2 | 波平・フネ |
| ワカメ | 2 | サザエ・カツオ | 2 | 波平・フネ |
| マスオ | 1 | サザエ（妻） | 1 | タラオ |
| タラオ | 0 | いない | 2 | サザエ・マスオ |

**同じ家族なのに、人によって数字が違う。** 全員が7人家族だが `sibsp` は 0〜3、
`parch` は 1〜3 とばらける。自分から見た関係を数えているため。

マスオから見た波平・フネ（妻の親）を数えるかどうかは、データの説明に書かれていない。
判断できないので、この表では除いてある。

### 数字を見ても、誰を数えたかは分からない

上の表は「値がこう決まる」という説明で、**手元のデータからは逆にたどれない**。
入っているのは合計の数字だけ。

| 見える数字 | 分からないこと |
| --- | --- |
| `sibsp = 1` | 配偶者なのか、兄弟姉妹なのか |
| `parch = 2` | 親2人なのか、子2人なのか、親と子1人ずつなのか |
| `parch = 3`（サザエ） | 親と子が混ざっていること自体 |

これが効いてくるのは、**子どもは優先して助けられたと言われている**ため。
`parch = 2` の人が「親に連れられた子ども」なら助かりやすく、
「子どもを連れた親」なら立場が違うはずだが、数字だけでは区別できない。

ただし `age` と組み合わせれば見当は付く。

In [ ]:
# parch が 2 の人を年齢で分けてみる
two = train[train["parch"] == 2]

print("parch が 2 の人:", len(two), "人")
print("  10 歳未満  :", len(two[two["age"] < 10]), "人 → 親に連れられた子どもか")
print("  30 歳以上  :", len(two[two["age"] >= 30]), "人 → 子どもを連れた親か")
print("  年齢が空き :", two["age"].isna().sum(), "人 → 見当も付かない")

### 他の項目で家族を見分けられないか

`sibsp` と `parch` は自分から見た人数なので、**誰と誰が同じ家族なのかは分からない**。

氏名やチケット番号の列があれば、姓が同じ人や同じ番号の人を家族として結びつけられる。
しかしこのデータには入っていない。`id` は順番に振られた番号でしかない。

残る手がかりは **`fare`（運賃）** 。
家族でまとめてチケットを買っていれば、同じ金額が並ぶはず。
まずは運賃が何を表しているのかを確かめる。

## 5. `fare` は一人分か、まとめて1枚分か

運賃が「その人1人の料金」なのか「同乗した家族の合計」なのかは、説明に書かれていない。
どちらかで意味が大きく変わるので確かめる。

**調べ方**: 合計なら、家族の人数が多いほど運賃も大きくなるはず。

客室の等級は3等に固定して比べる。等級が混ざると、
運賃の差が「家族の人数」のせいか「客室の良さ」のせいか分からなくなる。

In [ ]:
# 同乗した家族の人数。自分を足すので +1
train["family_size"] = train["sibsp"] + train["parch"] + 1

third = train[train["pclass"] == 3].copy()
third["fare_per_person"] = third["fare"] / third["family_size"]

third.groupby("family_size").agg(
    人数=("fare", "count"),
    運賃の中央値=("fare", "median"),
    一人あたり=("fare_per_person", "median"),
).round(2)

### 分かったこと

運賃の中央値は、家族1人の **7.87** から 11 人の **69.55** まで増えていく。
一人あたりに割ると、どの人数でも **4〜8** の範囲に収まる。

つまり `fare` は **1枚のチケットの合計額**で、同じ値が同乗者全員の行に入っている。
ディズニーのチケットで言えば、家族全員分をまとめて買った金額が、
一人ひとりの記録に同じように書かれている状態。

### そのまま使うと起きること

**大家族の人が「高い運賃を払った人」として扱われる。**

3等客室の11人家族は運賃 69.55。これは1等客室の人と並ぶ金額だが、
一人あたりに直すと 6.32 で、3等の単身客（7.87）より安い。

運賃を豊かさの目安として使いたいなら、家族の人数で割った方が実態に近い。

## 6. 運賃を手がかりに家族を探す

運賃が1枚分の合計だと分かったので、**同じ金額の人は同じチケットを買った可能性**がある。
まず、同額の人がどれくらいいるのかを見る。

In [ ]:
fare_counts = train["fare"].value_counts()

print("同額が 2 人以上いる運賃:", (fare_counts >= 2).sum(), "種類")
print("その人数の合計:", fare_counts[fare_counts >= 2].sum(), "人 /", len(train), "人")
print()
print("人数が多い運賃:")
print(fare_counts.head(3).to_string())

# 8 割近くが誰かと同額。ただし家族でまとめて買った組だけでなく、
# 同じ料金区分を 1 人で買った人も混ざっている。
# 運賃が同じだけでは家族と決められない

### `sibsp` に 8 があって 6 と 7 が無い理由

3で気になっていた欠けを、ここで追う。

`sibsp = 8` の人を取り出し、**運賃を含めた他の項目が揃っているか**を見る。
揃っていれば、同じチケットで乗った1つの家族だと考えられる。

In [ ]:
big = train[train["sibsp"] == 8]

print("sibsp が 8 の人:", len(big), "人")
print("  parch     :", sorted(big["parch"].unique()), "→ 全員が親 2 人と同乗")
print("  客室の等級:", sorted(big["pclass"].unique()), "→ 全員同じ")
print("  運賃      :", sorted(big["fare"].unique()), "→ 全員同額")
print("  生死      :", big["survived"].tolist())

### 分かったこと

4つの項目が全員一致している。**同じチケットで乗った1つの家族**と考えるのが自然。

`sibsp = 8` は兄弟姉妹が8人、`parch = 2` は親2人なので、
自分を足して **11人家族**。運賃 69.55 はその11人分の合計。

**6 と 7 が無いのは、その人数の家族がたまたま乗っていなかっただけ。**
値が飛んでいるのはデータの不備ではない。

ここに出ている6人は**全員亡くなっている**。
3等客室の大家族で、救命ボートに全員分の席を確保できなかったことが想像できる。

### ただし断定はできない

運賃が同額でも、別々に買った他人という可能性は残る。
同額の人は他にも85種類・355人分いて、その多くは家族ではない。

この6人については条件が揃いすぎているので家族と考えて自然だが、
**氏名の列が無い以上、確かめる手段はない**。

## 7. 他の項目も使って家族を絞り込む

運賃だけでは、同じ金額を1人で買った人が混ざってしまう。
そこで**同じ家族なら揃っているはずの項目**を足して絞り込む。

| 項目 | 同じ家族なら |
| --- | --- |
| `fare` | まとめて買った1枚分なので同額 |
| `pclass` | 同じ客室の等級 |
| `embarked` | 同じ港から乗っている |

この3つが一致する人をひとまとめにして、
**その人数が本人の申告（`sibsp + parch + 1`）と合うか**を見る。
合っていれば、その家族全員を見つけられたことになる。

ここでは `train` と `test` を合わせた891人で調べる。
家族が2つのファイルに分かれて入っているため、片方だけでは人数が足りない。

In [ ]:
test = pd.read_csv("../../data/raw/test.csv", index_col=0)
both = pd.concat([train, test])

# 本人が申告した家族の人数
both["family_size"] = both["sibsp"] + both["parch"] + 1

# 運賃・等級・港 が一致する人のかたまりの大きさ
#   transform("size") は、各行に「自分が属するグループの人数」を書き込む
both["group_size"] = both.groupby(
    ["fare", "pclass", "embarked"], dropna=False
)["sibsp"].transform("size")

matched = both[both["group_size"] == both["family_size"]]

print("全体:", len(both), "人")
print("グループの人数が申告と一致:", len(matched), "人")

# 出力の見方
#   一致した人は、家族全員を見つけられたと考えられる
#   一致しないのは、同額の他人が混ざっている / 家族の一部しか名簿に無い などの理由

### 見つけた家族を見てみる

一致した家族の1つを、年齢の高い順に並べる。
`sex`・`age`・`sibsp`・`parch` を横に並べると、**誰が親で誰が子かが読み取れる**。

In [ ]:
family = both[
    (both["fare"] == 27.9) & (both["pclass"] == 3) & (both["embarked"] == "S")
].sort_values("age", ascending=False)

family[["sex", "age", "sibsp", "parch", "family_size"]]

# 出力の見方
#   45 歳の女性と 40 歳の男性 … sibsp=1（配偶者）, parch=4（子 4 人）→ 両親
#   10, 9, 4, 2 歳の 4 人    … sibsp=3（兄弟 3 人）, parch=2（親 2 人）→ 子ども
#
#   6 人全員の family_size が 6 で、グループの人数とも一致する。
#   両親 2 人 + 子ども 4 人の 6 人家族を、丸ごと特定できた

### 読み取り方のまとめ

`sibsp` と `parch` の組み合わせで、立場が見分けられる。

| `sibsp` | `parch` | 年齢 | 立場 |
| --- | --- | --- | --- |
| 1 | 子の数 | 高い | **親**（配偶者1人 + 子ども） |
| 兄弟の数 | 2 | 低い | **子ども**（兄弟 + 親2人） |
| 0 | 子の数 | 高い | **ひとり親**（配偶者が乗っていない） |
| 0 | 0 | — | **単身** |

`parch = 2` を見ただけでは親か子か分からなかったが、
`sibsp` と `age` を並べれば区別が付く。

### ただし使える範囲は限られる

| | 人数 |
| --- | --- |
| 全体 | 891 人 |
| グループの人数が申告と一致した人 | **184 人（20.7%）** |
| 年齢が空いている人 | 177 人 |
| そのうち家族を特定できた人 | 28 人 |
| さらに家族の年齢から立場を推測できる人 | **3 人** |

**年齢の空きを埋める用途には、ほとんど使えない。**
家族を特定できるのは全体の2割で、年齢が空いている人まで届くのは3人だけ。

一方で、**特定できた2割については立場がはっきり分かる**。
「親か子か」を表す項目を新しく作るなら、この2割に正しい値を入れられる。

## 8. なぜ2割しか特定できないのか

一致したのは 184 人だけで、残り 707 人は合わなかった。理由は2つある。

In [ ]:
too_many = (both["group_size"] > both["family_size"]).sum()
too_few = (both["group_size"] < both["family_size"]).sum()

print("グループの人数 > 申告:", too_many, "人 → 他人が混ざっている")
print("グループの人数 < 申告:", too_few, "人 → 家族が名簿に載っていない")
print()

# 人数の多い運賃グループの中身を見る
for fare in [8.05, 13.0, 7.75]:
    group = both[both["fare"] == fare]
    print(f"運賃 {fare}: {len(group)} 人 / 申告した家族の人数 {sorted(group['family_size'].unique())}")

### 理由1: 同じ運賃を別々に買った他人がいる（590人）

運賃は**料金表の値段**なので、同じ条件で1人ずつ買えば同額になる。

```
運賃 8.05 : 43 人 → 全員が「家族の人数 1」＝ 全員が単身
運賃 13.0 : 42 人 → 単身の人と3人家族が混ざっている
運賃 7.75 : 34 人 → 単身・2人家族・3人家族が混ざっている
```

`8.05` の43人は**全員が単身者**。3等の標準的な運賃だったので、
無関係な43人が同じ金額になっている。これをひとまとめにすると43人家族に見えてしまう。

ディズニーのチケットで、大人1枚を別々に買った人同士が同じ金額になるのと同じ。

### 理由2: 家族が名簿に載っていない（117人）

**このデータは全乗客ではない。** 実際の乗客は1300人以上いるが、
ここにあるのは 891 人分。家族の一部しか収録されていない人がいる。

「兄弟が3人いる」と申告しているのに名簿上は2人しかいない、という状態。
この場合は何をしても特定できない。

## 9. 家族を特定できなくても、家族構成は使える

ここまでの結果を見ると、家族を1組ずつ特定する作業は**割に合わない**。
2割しかできず、名簿に載っていない人は原理的に無理。

ただしこれは「家族構成の情報が使えない」という話ではない。
**特定しなくても作れる項目がある。**

### 家族の人数

`sibsp + parch + 1` は**本人の申告**なので、
相手が名簿にいるかどうかに関係なく計算できる。

In [ ]:
# family_size は 5 で作ったものをそのまま使う
train.groupby("family_size")["survived"].agg(
    生存率="mean", 人数="count"
).round(3)

# 出力の見方
#   全体の生存率は 0.402。それと比べて読む
#   単身（1 人）  0.305 → 不利
#   2〜4 人       0.6 前後 → 有利
#   5 人以上      0.33 以下。8 人と 11 人は 0.000 → 急に不利
#
#   山なりの形になっている。sibsp と parch を別々に数値で渡すだけでは、
#   この形は表現できない。足して人数にすることに意味がある

### 立場（親か子か）

これも本人の行だけで判定できる。
「年齢が15歳未満で `parch` が1以上なら、親に連れられた子」という具合。

家族を特定する必要はない。

In [ ]:
def guess_role(row):
    """本人の行だけを見て、家族の中での立場を推測する"""
    if row["family_size"] == 1:
        return "単身"
    if pd.isna(row["age"]):
        return "年齢不明"
    if row["age"] < 15 and row["parch"] > 0:
        return "親に連れられた子"
    if row["age"] >= 20 and row["parch"] > 0 and row["sibsp"] <= 1:
        return "子を連れた親"
    return "その他"


train["role"] = train.apply(guess_role, axis=1)

train.groupby("role")["survived"].agg(生存率="mean", 人数="count").round(3)

# 出力の見方
#   親に連れられた子 0.583 / 子を連れた親 0.490 / 単身 0.305
#   子どもが優先して助けられたという話が、数字にも出ている

### 使い分け

| やること | 意味 |
| --- | --- |
| 誰と誰が同じ家族かを特定する | **割に合わない。** 2割しかできず、名簿に載っていない人は無理 |
| **家族の人数を項目にする** | **有効。** 全員分作れて、生存率にはっきり差が出る |
| **立場（親か子か）を項目にする** | **有効。** 本人の行だけで判定でき、差も出る |

家族の特定は「年齢の空きを埋める手段」として考えていたが、
そこは3人しか届かないので諦めてよい。

一方で家族構成を表す項目を作ることは別の話で、こちらは特定作業なしにできる。

## この章のまとめ

**説明に書かれていたこと**

- 8つの項目の意味。`sibsp` は兄弟姉妹と配偶者、`parch` は親と子の人数

**調べて分かったこと**

- `age` に 85 件、`embarked` に 2 件、値が入っていない人がいる
- `sex` と `embarked` は文字なので、そのままでは計算に使えない
- `sibsp` と `parch` は合計の数字だけで、**誰を数えたかは分からない**
- **`fare` は1枚のチケットの合計額**。同じ値が同乗者全員の行に入っている
- 運賃が同額でも家族とは決められないが、他の項目まで揃えば推測はできる。
  `sibsp` に 8 があるのは 11人家族が乗っていたため
- **運賃・等級・港が一致する人をまとめると、891人中 184人（2割）は家族を特定できる**。
  その人たちは `sibsp` `parch` `age` の組み合わせで、親か子かまで読み取れる
- 残り8割が特定できない理由は2つ。**同じ運賃を別々に買った他人が混ざる**（590人）と、
  **家族が名簿に載っていない**（117人）。このデータは全乗客ではなく 891 人分しかない
- ただし**家族を特定しなくても、家族構成の情報は項目にできる**。
  家族の人数（単身 0.305 / 2〜4人 0.6前後 / 5人以上は 0.33 以下で、8人と11人は 0.000）も、
  立場の推測（親に連れられた子 0.583 / 単身 0.305）も、本人の行だけで作れる

**予測に使うときに考えること**

| 項目 | 考えること |
| --- | --- |
| `age` `embarked` | 空いている場所をどう埋めるか |
| `sex` `embarked` | 文字をどう数字に置き換えるか |
| `pclass` | 数字は等級の順位。1と2の差に「1人分」のような意味はない |
| `sibsp` `parch` | 足して家族の人数にする、`age` と組み合わせるなど |
| `fare` | 家族の人数で割って一人あたりにするか |